# Vežba 2: Priprema podataka i Pipeline

Ponovljeni zapisi, neispravne vrednosti, kodiranje i skaliranje, i redosled pripreme koji sprečava curenje podataka

Lekcija sa objašnjenjima: [https://isatvss.surge.sh/vezba2.html](https://isatvss.surge.sh/vezba2.html)

Ćelije se izvršavaju redom, od vrha nadole (Shift+Enter), jer svaka koristi promenljive iz prethodnih.

## Jedan red po nezgodi

### Zadatak 1. Učitavanje i obeležja sa vežbe 1

Učitati skup nezgoda i ponoviti izvođenje ciljne promenljive i obeležja iz vremena sa vežbe 1, da bi sveska radila samostalno.

In [ ]:
import numpy as np
import pandas as pd

nezgode = pd.read_csv("https://isatvss.surge.sh/podaci/nezgode_2025.csv")
nezgode["teska"] = (nezgode["posledica"] != "Sa mat.stetom").astype(int)

vreme = pd.to_datetime(nezgode["datum_vreme"], format="%d.%m.%Y,%H:%M")
nezgode["sat"] = vreme.dt.hour
nezgode["dan_u_nedelji"] = vreme.dt.dayofweek
nezgode["mesec"] = vreme.dt.month

print("Redova:", len(nezgode))

### Zadatak 2. Nezgode upisane u više redova

Izbrojati koliko je nezgoda upisano u jedan, dva i tri reda, proveriti da li ista nezgoda ikad ima različitu posledicu, pa zadržati po jedan red za svaku nezgodu.

Jedinica posmatranja je nezgoda, a ne red u tabeli. Kada bi dva reda iste nezgode završila jedan u treningu, a drugi u testu, model bi na testu procenjivao nezgodu koju je već video, što je drugi oblik curenja iz tabele. Zadržavanjem prvog reda gubi se druga vrsta, odnosno opis, kod 214 nezgoda sa dva i 7 nezgoda sa tri reda. To je mala i svesno prihvaćena cena, koja se navodi u izveštaju. Druga mogućnost je zadržati sve redove i deliti skup po nezgodi, a ne po redu (`GroupShuffleSplit`).

In [ ]:
redova_po_nezgodi = nezgode.groupby("id_nezgode").size()
print("Broj nezgoda prema broju redova:")
print(redova_po_nezgodi.value_counts())

razlicita_posledica = (nezgode.groupby("id_nezgode")["posledica"].nunique() > 1).sum()
print("\nNezgoda sa različitom posledicom u svojim redovima:", razlicita_posledica)

nezgode = nezgode.drop_duplicates(subset="id_nezgode", keep="first").reset_index(drop=True)
print("Redova posle zadržavanja jednog reda po nezgodi:", len(nezgode))

## Neispravne vrednosti i outlieri

### Zadatak 3. Koordinate koje ne pripadaju upravi

Pronaći nezgode čije koordinate očigledno ne odgovaraju mestu nezgode. Prvo proveriti pravougaonik oko Srbije, a zatim udaljenost svake nezgode od središta njene policijske uprave. Sumnjive koordinate označiti kao nedostajuće.

Pravougaonik oko Srbije otkriva samo 4 tačke, a provera po upravi njih 20. Kod 99 % nezgoda udaljenost od središta uprave ni u jednoj upravi ne prelazi 63,6 km, pa je granica od 100 km zadata sa velikom rezervom. Među pronađenim tačkama su nezgoda u Lazarevcu sa geografskom dužinom istočne Srbije i nezgode sa okruglim koordinatama, tačno 19 i 44 stepena, koje liče na podrazumevanu vrednost unetu umesto prave lokacije. Medijana se računa na svim podacima jer predstavlja geografsko središte uprave, koje bi se isto koristilo i za novu nezgodu.

In [ ]:
van_srbije = ~nezgode["lat"].between(42.2, 46.2) | ~nezgode["lon"].between(18.8, 23.1)
print("Tačaka van pravougaonika oko Srbije:", van_srbije.sum())

srediste = nezgode.groupby("policijska_uprava")[["lat", "lon"]].transform("median")
km_sever_jug = (nezgode["lat"] - srediste["lat"]) * 111
km_istok_zapad = (nezgode["lon"] - srediste["lon"]) * 111 * np.cos(np.radians(44))
nezgode["km_od_sredista"] = np.sqrt(km_sever_jug**2 + km_istok_zapad**2)

percentil_99 = nezgode.groupby("policijska_uprava")["km_od_sredista"].quantile(0.99)
print(f"Najveći 99. percentil udaljenosti po upravi: {percentil_99.max():.1f} km")

sumnjive = nezgode["km_od_sredista"] > 100
print("Tačaka dalje od 100 km od središta svoje uprave:", sumnjive.sum())
print(nezgode.loc[sumnjive, ["policijska_uprava", "opstina", "lon", "lat"]].round(2).to_string())

nezgode.loc[sumnjive, ["lon", "lat"]] = np.nan
nezgode = nezgode.drop(columns="km_od_sredista")
print("\nNedostajućih koordinata posle označavanja:", nezgode["lat"].isna().sum())

### Zadatak 4. Outlier koji nije greška

Na skupu o vazduhu primeniti pravilo interkvartilnog raspona (IQR) na koncentraciju PM10 i proveriti u kojim mesecima se javljaju dani iznad gornje granice.

Pravilo izdvaja 156 dana iznad 81,6 µg/m³, gotovo svi u zimskim mesecima, 51 u januaru i 46 u decembru. To nisu greške merenja, nego epizode zagađenja u sezoni grejanja. Brisanjem bi se iz skupa uklonili upravo dani koje model za kvalitet vazduha najviše treba da prepozna. Statistički outlier zato nije razlog za brisanje, nego za proveru.

In [ ]:
vazduh = pd.read_csv("https://isatvss.surge.sh/podaci/vazduh_nis.csv", parse_dates=["datum"])

q1, q3 = vazduh["pm10"].quantile([0.25, 0.75])
iqr = q3 - q1
gornja_granica = q3 + 1.5 * iqr
print(f"Q1 = {q1:.1f}, Q3 = {q3:.1f}, IQR = {iqr:.1f}, gornja granica = {gornja_granica:.1f}")

iznad = vazduh[vazduh["pm10"] > gornja_granica]
print("Dana iznad granice:", len(iznad))
print("\nPo mesecu:")
print(iznad["datum"].dt.month.value_counts().sort_index())

## Kodiranje i skaliranje

### Zadatak 5. One-hot kodiranje

Na prva četiri reda kolone `vrsta` pokazati kako `OneHotEncoder` pretvara kategoriju u kolone sa vrednostima 0 i 1.

Kategorije bi se mogle zameniti i brojevima 0, 1, 2 i tako dalje, ali bi model tada zaključivao da je kategorija 2 dvostruko veća od kategorije 1, odnosno da su kategorije 0 i 1 bliže nego 0 i 2. Za nominalne kategorije, bez prirodnog poretka, takvo kodiranje uvodi rastojanja kojih u podacima nema. To je posebno štetno kod metoda zasnovanih na rastojanju, kao što su kNN i k-means.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

primer = nezgode[["vrsta"]].head(4)
koder = OneHotEncoder(sparse_output=False)
kodirano = koder.fit_transform(primer)

print(primer.to_string())
print()
print(pd.DataFrame(kodirano, columns=koder.get_feature_names_out(), dtype=int).to_string())

### Zadatak 6. Standardizacija

Uporediti srednju vrednost i standardnu devijaciju numeričkih obeležja pre i posle standardizacije.

Bez skaliranja obeležje sa velikim rasponom, kao sat (od 0 do 23), dominira nad obeležjem sa malim, kao geografska dužina, čija je standardna devijacija 0,68. To utiče na metode zasnovane na rastojanju i na gradijentnom spustu (kNN, logistička regresija, neuronske mreže). Stabla odlučivanja porede svako obeležje posebno, pa na skaliranje nisu osetljiva.

In [ ]:
from sklearn.preprocessing import StandardScaler

numericke = ["sat", "dan_u_nedelji", "mesec", "lon", "lat"]
print(nezgode[numericke].describe().loc[["mean", "std"]].round(2))

skaliranje = StandardScaler()
skalirano = pd.DataFrame(skaliranje.fit_transform(nezgode[numericke]), columns=numericke)
print()
print(skalirano.describe().loc[["mean", "std"]].round(2))

## Podela i Pipeline

### Zadatak 7. Podela na trening i test skup

Izabrati obeležja i podeliti skup u odnosu 80:20, tako da udeo teških nezgoda bude isti u oba dela.

In [ ]:
from sklearn.model_selection import train_test_split

obelezja = ["sat", "dan_u_nedelji", "mesec", "lon", "lat", "policijska_uprava", "vrsta", "opis"]
X = nezgode[obelezja]
y = nezgode["teska"]

X_trening, X_test, y_trening, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

print("Trening:", X_trening.shape, f"udeo teških {y_trening.mean():.3f}")
print("Test:   ", X_test.shape, f"udeo teških {y_test.mean():.3f}")

### Zadatak 8. ColumnTransformer, Pipeline i prvi model

Spojiti popunjavanje i skaliranje numeričkih obeležja i kodiranje kategoričkih u jedan korak pripreme, dodati logističku regresiju i uporediti tačnost sa osnovnim modelom.

Posle pripreme svaka nezgoda opisana je sa 105 kolona, pet numeričkih i ostale iz one-hot kodiranja. Logistička regresija postiže tačnost 0,747, a osnovni model 0,594. Model, dakle, iz podataka izvlači stvarnu informaciju. Da li je tačnost prava mera za ovaj problem, razmatra se na vežbi 6.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

kategoricke = ["policijska_uprava", "vrsta", "opis"]

priprema = ColumnTransformer([
    ("numericke", Pipeline([("popuna", SimpleImputer(strategy="median")), ("skaliranje", StandardScaler())]), numericke),
    ("kategoricke", OneHotEncoder(handle_unknown="ignore"), kategoricke),
])

model = Pipeline([("priprema", priprema), ("klasifikator", LogisticRegression(max_iter=1000))])
model.fit(X_trening, y_trening)

osnovni = DummyClassifier(strategy="most_frequent").fit(X_trening, y_trening)

print("Broj kolona posle pripreme:", model.named_steps["priprema"].transform(X_trening.head(1)).shape[1])
print(f"Tačnost osnovnog modela na testu:     {osnovni.score(X_test, y_test):.3f}")
print(f"Tačnost logističke regresije na testu: {model.score(X_test, y_test):.3f}")

## Curenje podataka u praksi

### Zadatak 9. Izjednačavanje klasa pre i posle podele

Izjednačiti broj teških i lakih nezgoda kopiranjem teških, jednom pre podele, a jednom samo na trening skupu. U oba slučaja istrenirati stablo odlučivanja i uporediti tačnost na testu.

Kada se kopira pre podele, 34,2 % test redova ima identičnu kopiju u treningu. Stablo odlučivanja te redove zapamti, pa je tačnost 0,818. Kada se kopira samo trening skup, tačnost je 0,672. Razlika potiče isključivo od načina pripreme, a ne od kvaliteta modela. Ova greška se često sreće u projektima i lako se prepoznaje: rezultat je neobično dobar, a model na novim podacima radi lošije.

In [ ]:
from sklearn.base import clone
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils import resample

stablo = Pipeline([("priprema", clone(priprema)), ("klasifikator", DecisionTreeClassifier(random_state=42))])


def izjednaci_klase(X_deo, y_deo):
    deo = X_deo.assign(teska=y_deo)
    lake = deo[deo["teska"] == 0]
    teske = deo[deo["teska"] == 1]
    teske_uvecane = resample(teske, n_samples=len(lake), replace=True, random_state=42)
    izjednaceno = pd.concat([lake, teske_uvecane])
    return izjednaceno[obelezja], izjednaceno["teska"]


X_sve, y_sve = izjednaci_klase(X, y)
Xp_trening, Xp_test, yp_trening, yp_test = train_test_split(X_sve, y_sve, test_size=0.2, stratify=y_sve, random_state=42)
stablo.fit(Xp_trening, yp_trening)
kopije = Xp_test.index.isin(Xp_trening.index)
print("POGREŠNO: izjednačavanje pre podele")
print(f"  test redova koji postoje i u treningu: {kopije.mean():.1%}")
print(f"  tačnost na testu: {stablo.score(Xp_test, yp_test):.3f}")

Xi_trening, yi_trening = izjednaci_klase(X_trening, y_trening)
stablo.fit(Xi_trening, yi_trening)
X_test_izj, y_test_izj = izjednaci_klase(X_test, y_test)
print("ISPRAVNO: podela, pa izjednačavanje samo trening skupa")
print(f"  tačnost na test skupu izjednačenom na isti način: {stablo.score(X_test_izj, y_test_izj):.3f}")

## Zadaci za samostalan rad

### Zadatak 10. Nedostajuća merenja PM10

Na skupu o vazduhu izbrojati dane bez merenja PM10 po godini i pronaći najduži niz uzastopnih dana bez merenja. Zašto je dužina takvog niza važna za izbor načina popunjavanja?

### Zadatak 11. Medijana ili interpolacija

Sakriti 150 nasumično izabranih poznatih vrednosti PM10, popuniti ih medijanom i linearnom interpolacijom između susednih dana, pa za oba načina izračunati prosečnu apsolutnu grešku u odnosu na stvarne vrednosti. Koji način je bolji i zašto?

### Zadatak 12. Retke kategorije

Izbrojati opise nezgoda u trening skupu koji se javljaju manje od 50 puta. Zatim uporediti broj kolona koje daje `OneHotEncoder` sa podrazumevanim podešavanjima i sa parametrom `min_frequency=50`, koji retke kategorije spaja u jednu kolonu.

### Zadatak 13. Podela vremenske serije

Za regresiju PM10 na vežbama 3 i 4 skup o vazduhu podeliti po vremenu: dani do kraja 2022. su trening, a 2023. i 2024. test. Napraviti `ColumnTransformer` koji meteorološke kolone popunjava i skalira, a mesec kodira one-hot postupkom, naučiti ga na treningu i ispisati oblik pripremljene test matrice. Zašto nasumična podela ovde ne bi bila poštena?

### Zadatak 14. Da li koordinate pomažu

Istrenirati isti model kao u zadatku 8, ali bez geografskih koordinata, i uporediti tačnost na testu. Šta rezultat govori o vrednosti obeležja za ovaj model?